# 001 LangChain Overview And Agent

这是 LangChain 学习线的第一份 Notebook。

官方参考：

- https://docs.langchain.com/oss/python/langchain/overview
- https://docs.langchain.com/oss/python/langchain/install
- https://docs.langchain.com/oss/python/langchain/agents
- https://docs.langchain.com/oss/python/langchain/tools

学习目标：

1. 理解 LangChain 在 agent 应用里负责什么
2. 安装并导入 LangChain v1 相关包
3. 用 `create_agent` 创建一个最小 agent
4. 了解 model、tools、system prompt、messages 的关系
5. 把 LangChain 的概念映射到本仓库已有的 Harness runtime

---

## 1. 先建立心智模型

LangChain 可以先理解为一组构建 LLM 应用的标准组件。

在第一阶段，不要把它想成一个神秘框架。先把它拆成四件事：

| LangChain 概念 | 先这样理解 | 本仓库对应概念 |
|---|---|---|
| model | 模型调用接口 | `ModelClient` |
| messages | 对话上下文 | `history_snapshot` / `completed_steps` |
| tools | 模型可调用的外部能力 | `ToolRegistry` |
| agent | 模型 + 工具 + 控制流 | `HarnessChatAgent` |

LangChain 的优势是：它把这些常见部件标准化了。我们自己写 Harness runtime 的优势是：可以更清楚地学习权限、审批、ledger、subagent 隔离这些工程边界。

## 2. 安装依赖

LangChain v1 的核心包和 OpenAI 集成包分开安装。

如果你已经安装过，可以跳过这一格。

In [2]:
%pip install -U langchain langchain-openai python-dotenv

/home/dev/bxc/fastapi-study/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


## 3. 加载项目环境变量

复用本项目根目录 `.env`：

```env
OPENAI_API_KEY=your-key
OPENAI_MODEL=gpt-5.4-mini
OPENAI_BASE_URL=
```

如果你使用 OpenAI 兼容网关，`OPENAI_BASE_URL` 可以保留网关地址。

In [5]:
import os
from pathlib import Path

from dotenv import load_dotenv


def load_project_env() -> Path | None:
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        env_path = path / ".env"
        if env_path.exists():
            load_dotenv(env_path, override=False)
            return env_path
    return None


env_path = load_project_env()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL") or None

print(f"Loaded .env: {env_path}" if env_path else "No .env found")
print("OPENAI_MODEL =", OPENAI_MODEL)
print("OPENAI_API_KEY loaded =", bool(OPENAI_API_KEY))
print("OPENAI_BASE_URL =", OPENAI_BASE_URL)

Loaded .env: /home/dev/bxc/fastapi-study/.env
OPENAI_MODEL = qwq
OPENAI_API_KEY loaded = True
OPENAI_BASE_URL = http://192.168.102.19:8082/v1


## 4. 导入 LangChain

第一讲只用三个核心对象：

- `create_agent`：创建 agent
- `tool`：把普通 Python 函数声明成工具
- `ChatOpenAI`：OpenAI 或 OpenAI 兼容模型客户端

注意：`ChatOpenAI` 来自 `langchain-openai`，不是 LangChain 核心包。

In [7]:
try:
    from langchain.agents import create_agent
    from langchain_core.tools import tool
    from langchain_openai import ChatOpenAI
    print("LangChain imports OK")
except ImportError as exc:
    print("LangChain dependency is missing. Run the install cell first.")
    raise

LangChain imports OK


## 5. 定义一个最小 Tool

LangChain tool 可以来自普通 Python 函数。函数名、参数类型和 docstring 都会影响模型理解这个工具。

这和本仓库的 `ToolDefinition` 很像：都需要名称、说明、参数和执行函数。

In [8]:
@tool
def explain_repo_concept(topic: str) -> str:
    """Explain one learning concept from this FastAPI agent study repo."""
    concepts = {
        "harness": "Harness 负责控制流、权限、上下文、ledger 和恢复，不只是 prompt。",
        "tool": "Tool 是受管执行接口，模型只能提出调用，系统负责权限和执行。",
        "subagent": "Subagent 是隔离的受管 worker，只拿任务切片，不共享完整主上下文。",
        "skill": "Skill 是可复用工作流知识包，通常包含 SKILL.md、references 或 scripts。",
    }
    key = topic.strip().lower()
    return concepts.get(key, f"暂时没有 {topic!r} 的固定解释，可以换成 harness、tool、subagent 或 skill。")


print(explain_repo_concept.invoke({"topic": "harness"}))

Harness 负责控制流、权限、上下文、ledger 和恢复，不只是 prompt。


## 6. 创建 Chat Model

这里显式创建 `ChatOpenAI`，而不是只传字符串模型名，原因是本项目可能使用 OpenAI 兼容网关，需要传入 `base_url`。

In [9]:
if not OPENAI_API_KEY:
    print("OPENAI_API_KEY is not configured. Configure .env before running live agent calls.")
    chat_model = None
else:
    chat_model = ChatOpenAI(
        model=OPENAI_MODEL,
        api_key=OPENAI_API_KEY,
        base_url=OPENAI_BASE_URL,
    )
    print("Chat model ready:", OPENAI_MODEL)

Chat model ready: qwq


## 7. 创建第一个 Agent

`create_agent` 会把模型、工具和系统提示词组合成一个可调用对象。

这里的 agent 和我们自己写的 `HarnessChatAgent` 有相似点：

- 都会接收用户消息
- 都能根据需要调用工具
- 都会把工具结果再交给模型组织回答

区别是：本仓库 Harness 显式暴露 ledger、approval、subagent policy；LangChain 把更多通用控制流封装起来。

In [10]:
if chat_model is None:
    agent = None
    print("Skip create_agent because chat_model is not configured.")
else:
    agent = create_agent(
        model=chat_model,
        tools=[explain_repo_concept],
        system_prompt="你是一个面向 Java 开发者的中文教学助手。回答要简洁，必要时使用工具。",
    )
    print("Agent ready")

Agent ready


## 8. 调用 Agent

LangChain agent 的输入通常是 messages。先看最小调用：

In [14]:
def print_last_message(result):
    messages = result.get("messages", []) if isinstance(result, dict) else []
    if not messages:
        print(result)
        return
    last = messages[-1]
    content = getattr(last, "content", last.get("content") if isinstance(last, dict) else str(last))
    print(content)


if agent is None:
    print("Skip live invoke because agent is not configured.")
else:
    result = agent.invoke({
        "messages": [
            {"role": "user", "content": "请解释这个项目里的 subagent 是什么"}
        ]
    })
    print_last_message(result)

APIConnectionError: Connection error.

## 9. 观察：LangChain 帮我们隐藏了什么

如果这次调用触发了工具，LangChain 会替你处理一部分 agent 循环。

但在真实工程里，你仍然要思考：

1. 工具能不能自动执行？
2. 高风险工具是否需要审批？
3. 工具结果是否需要独立验证？
4. 上下文是否需要裁剪？
5. 失败后怎么恢复？

这也是本仓库 Harness 学习线的价值：它让这些控制面显性化。

## 10. 本讲小结

这一讲先记住一个最小公式：

```text
LangChain agent = model + tools + system_prompt + messages + agent loop
```

和本仓库对照：

```text
HarnessChatAgent = ModelClient + ToolRegistry + planning_prompt + ledger + permission + recovery
```

下一讲建议学习：

- model 统一接口
- message 类型
- 多轮上下文
- 为什么 LangChain 也需要上下文治理